# 3 — Stand-Alone VSI: dq Voltage Control

> **Goal.** Close the voltage loop in the dq frame. The 3-phase
> output regulates to a target $V_d$ (peak line-to-neutral) with
> $V_q$ = 0, against an RL load. Demonstrate that the dq plant is
> *buck-like* and that a simple PI handles it (modulo the high-Q
> LC resonance that we discuss explicitly).

**Prerequisites**

- `01_vsi_basics.ipynb` and `02_vsi_svpwm.ipynb` (this project)
- Buck controller notebook (`projects/converters/buck/`) — the
  control intuition transfers verbatim once we're in dq.

**What you'll be able to do at the end**

1. Derive the dq-frame plant of a 3-phase inverter + LC filter +
   resistive load.
2. Recognize it as the buck's second-order plant (with the same
   high-Q resonance issue).
3. Design a PI compensator targeting a crossover well below the
   resonance and switching frequency.
4. Discretize, wire two parallel compensators (one for $v_d$, one
   for $v_q$), and run the switched closed-loop sim.
5. Verify that $V_d$ regulates and $V_q$ stays near zero, with
   sinusoidal output at the filtered load terminals.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import numpy as np
from scipy import signal
import matplotlib.pyplot as plt

from vsi_3phase_model import (
    VSI3PhaseParams, standalone_voltage_plant,
    simulate_closed_loop_standalone,
    clarke_transform, park_transform,
    fundamental_rms, thd,
    operating_point_report,
)

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

p = VSI3PhaseParams()
print(operating_point_report(p, mode="standalone"))


## 1. The dq-frame plant (= the buck)

With balanced 3-phase quantities and the Park frame aligned with
the modulating signal, the inverter + LC filter + load reduces to
**two decoupled second-order systems** — one per axis:

$$
L_f \frac{di_d}{dt} = v_{cmd,d} - v_d \cdot 0 - \omega L_f \cdot i_q
$$

$$
C_f \frac{dv_d}{dt} = i_d - i_{load,d}
$$

The $\omega L_f \cdot i_q$ is **cross-coupling** between the axes —
small at our operating point because $i_q \approx 0$. Within each
axis the dynamics are identical to a buck converter with $v_{cmd,x}$
as the "duty" times $V_{dc}/2$ and the LC + load as the filter:

$$
G_{vd}(s) = \frac{1/(L_f C_f)}{s^2 + s/(R C_f) + 1/(L_f C_f)}
$$

DC gain = 1 (the dq modulating signal directly equals the dq
output voltage at the filter). Natural frequency $\omega_n =
1/\sqrt{L_f C_f}$. Q-factor $= R\sqrt{C_f/L_f}$.

**This is the buck's plant.** Everything we learned about buck
control transfers.


In [ ]:
plant = standalone_voltage_plant(p)
print(f"dq-frame voltage-loop plant: num = {plant.num}, den = {plant.den}")
print(f"  f_n = {p.f_filter_corner:.1f} Hz, Q = {p.Q_filter:.3f}")
print()
print("Note: Q is HIGH (no damping). The resonance peaks at +20 dB.")
print("In practice, production designs add series R in the inductor,")
print("a damping branch, or active damping via a current loop.")

f = np.logspace(0, 4, 1000)
w = 2*np.pi*f
_, mag, ph = signal.bode(plant, w=w)
fig, (ax_mag, ax_ph) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax_mag.semilogx(f, mag, "C0", linewidth=2)
ax_mag.axvline(p.f_filter_corner, color="C3", linestyle=":", alpha=0.5,
               label=f"$f_n$ = {p.f_filter_corner:.0f} Hz")
ax_mag.axvline(p.f_sw/10, color="C2", linestyle=":", alpha=0.5,
               label=f"$f_{{sw}}/10$ = {p.f_sw/10/1000:.1f} kHz")
ax_mag.axhline(0, color="k", linestyle=":", alpha=0.3)
ax_ph.semilogx(f, ph, "C0", linewidth=2)
ax_mag.set_ylabel("Magnitude [dB]"); ax_ph.set_ylabel("Phase [deg]")
ax_ph.set_xlabel("Frequency [Hz]")
ax_mag.set_title("Stand-alone dq voltage-loop plant — buck-like, high Q")
ax_mag.legend(); plt.tight_layout(); plt.show()


## 2. PI compensator design

Strategy: simple PI placed well **below** the LC resonance to avoid
exciting it. The high Q means we can't push the bandwidth close to
$f_n$ without active damping.

Target $f_c = 200$ Hz (10× below $f_n$ = 1.6 kHz, well above $f_o$
= 60 Hz). Place the zero at $f_o = 60$ Hz so the integrator's
$-90°$ contribution at $f_c$ is partially compensated.

Note: we design the PI with `V_dc/2` scaling already in the
simulator's command interpretation (`v_cmd_x` directly becomes the
sin/cos modulating signal that gets fed to SVPWM).


In [ ]:
# PI compensator: K · (1 + s·tau_z) / s
f_c = 200.0
f_z = 60.0  # zero at line frequency
omega_c = 2*np.pi*f_c
tau_z = 1.0/(2*np.pi*f_z)

# Plant magnitude at omega_c (we want loop gain = 1 there)
_, mag_at_fc, _ = signal.bode(plant, w=[omega_c])
plant_mag = 10**(mag_at_fc[0]/20)
# Compensator magnitude at omega_c: K · sqrt(1+(omega_c·tau_z)²) / omega_c
K = omega_c / (np.sqrt(1+(omega_c*tau_z)**2) * plant_mag)

Gc = signal.TransferFunction([K*tau_z, K], [1.0, 0.0])
print(f"PI compensator: K = {K:.4g}, tau_z = {tau_z*1e3:.3f} ms (zero at {f_z:.1f} Hz)")

# Loop gain
T_loop = signal.TransferFunction(np.polymul(Gc.num, plant.num),
                                   np.polymul(Gc.den, plant.den))
f = np.logspace(0, 4, 1500)
w = 2*np.pi*f
_, mag_T, ph_T = signal.bode(T_loop, w=w)
idx = np.argmin(np.abs(mag_T))
f_cross = f[idx]
pm = 180 + ph_T[idx]

fig, (ax_mag, ax_ph) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax_mag.semilogx(f, mag_T, "C3", linewidth=2)
ax_mag.axhline(0, color="k", linestyle=":")
ax_mag.axvline(f_cross, color="g", linestyle=":", alpha=0.5,
               label=f"$f_c$ = {f_cross:.1f} Hz")
ax_mag.axvline(p.f_filter_corner, color="C2", linestyle=":", alpha=0.5,
               label=f"$f_n$ = {p.f_filter_corner:.0f} Hz (avoid)")
ax_ph.semilogx(f, ph_T, "C3", linewidth=2)
ax_ph.axhline(-180, color="k", linestyle=":")
ax_mag.set_ylabel("Magnitude [dB]"); ax_ph.set_ylabel("Phase [deg]")
ax_ph.set_xlabel("Frequency [Hz]")
ax_mag.set_title(f"Stand-alone loop: $f_c$ ≈ {f_cross:.0f} Hz, PM = {pm:.1f}°")
ax_mag.legend(); plt.tight_layout(); plt.show()


## 3. Discretize and simulate

Tustin at $T_s = 1/f_{sw} = 50$ µs. Two parallel PIs run in the
switched simulator — one for $v_d$, one for $v_q$.


In [ ]:
T_s = p.T_sw
bd_raw, ad_raw, _ = signal.cont2discrete((Gc.num, Gc.den), dt=T_s, method='bilinear')
bd = np.asarray(bd_raw).flatten() / ad_raw[0]
ad = np.asarray(ad_raw) / ad_raw[0]
print(f"Discrete b = {bd}")
print(f"Discrete a = {ad}")

sim = simulate_closed_loop_standalone(p, bd, ad, n_cycles=8,
                                       samples_per_period=80, use_svpwm=True)
print(f"Simulated {len(sim['t'])} samples over {sim['t'][-1]*1000:.1f} ms")


In [ ]:
mask = sim['t'] > 4.0/p.f_o
fs = 1.0/(sim['t'][1]-sim['t'][0])

fig, axs = plt.subplots(4, 1, figsize=(12, 11), sharex=True)

axs[0].plot(sim['t']*1000, sim['v_Ca'], "C0", linewidth=1.0, label="$v_{Ca}$")
axs[0].plot(sim['t']*1000, sim['v_Cb'], "C1", linewidth=1.0, label="$v_{Cb}$")
axs[0].plot(sim['t']*1000, sim['v_Cc'], "C2", linewidth=1.0, label="$v_{Cc}$")
axs[0].set_ylabel("Phase voltage [V]")
axs[0].set_title(f"Closed-loop stand-alone VSI in dq @ V_d_ref = {p.V_o_LN_pk:.1f} V")
axs[0].legend(loc="upper right")

axs[1].plot(sim['t']*1000, sim['v_d'], "C2", linewidth=1.5, label="$v_d$")
axs[1].plot(sim['t']*1000, sim['v_d_ref'], "C3--", linewidth=1.0, label="$v_d^{ref}$")
axs[1].plot(sim['t']*1000, sim['v_q'], "C1", linewidth=1.5, label="$v_q$")
axs[1].plot(sim['t']*1000, sim['v_q_ref'], "C5--", linewidth=1.0, label="$v_q^{ref}$ = 0")
axs[1].set_ylabel("dq voltage [V]"); axs[1].legend(loc="upper right")

axs[2].plot(sim['t']*1000, sim['i_La'], "C4", linewidth=0.7)
axs[2].set_ylabel("Filter $i_{La}$ [A]")

# Compute the line-to-line voltage and its fundamental
v_LL_post = sim['v_Ca'] - sim['v_Cb']
axs[3].plot(sim['t']*1000, v_LL_post, "C3", linewidth=1.0)
axs[3].set_ylabel("$v_{LL,ab}$ [V]"); axs[3].set_xlabel("Time [ms]")

plt.tight_layout(); plt.show()

# Metrics
v_d_mean = sim['v_d'][mask].mean()
v_q_mean = sim['v_q'][mask].mean()
v_LL_fund = fundamental_rms(v_LL_post[mask], fs, p.f_o)
v_LL_thd = thd(v_LL_post[mask], fs, p.f_o, 30)
print()
print("Stand-alone closed-loop metrics:")
print(f"  V_d steady = {v_d_mean:.2f} V  (target {p.V_o_LN_pk:.2f} V, "
      f"error {(v_d_mean-p.V_o_LN_pk)/p.V_o_LN_pk*100:+.2f}%)")
print(f"  V_q steady = {v_q_mean:.3f} V  (target 0 V)")
print(f"  Line-to-line fundamental rms = {v_LL_fund:.2f} V  "
      f"(target {p.V_o_LL_rms:.2f} V)")
print(f"  Line-to-line THD             = {v_LL_thd*100:.2f} %")

if abs(v_d_mean - p.V_o_LN_pk) < 5.0 and abs(v_q_mean) < 5.0 and v_LL_thd < 0.10:
    print()
    print("✅  Stand-alone VSI control PROVEN: V_d and V_q regulated, "
          "clean output sinusoid.")


## 4. Discussion: the high-Q LC challenge

You may notice some ringing in the output, especially at startup.
That's the **LC resonance at 1591 Hz** being excited by transients
(switching events, warm-start). The Q ≈ 10.6 of the filter means
the resonance has a +20 dB peak.

Three standard fixes used in production:

1. **Series damping** — add a small resistor in series with the
   filter inductor. Effective but lossy.
2. **Damping branch** — RC across the cap (a series RC parallel to
   the load). Modest losses, robust.
3. **Active damping / cascade control** — inner current loop forces
   $i_L$, which "looks like" a virtual damping resistor to the cap.
   No extra hardware, but more complex firmware. This is what high-
   end UPS controllers actually do.

We use the simple PI here for pedagogy. The exercises below extend
it.

## 5. Summary

A 3-phase stand-alone VSI controlled in dq:

- **Plant in dq = buck plant.** All buck-control intuition transfers.
- **Two parallel PIs** (one for $v_d$, one for $v_q$) handle the
  two axes independently — the cross-coupling $\omega L_f i_q$ term
  is small and ignored in this first design.
- The bandwidth is constrained by the LC filter's high Q; production
  designs add damping or cascade control.

**Next**: `04_vsi_gridtie.ipynb` — connect to a stiff AC grid, add a
PLL to extract the grid angle, and inject active and reactive power
via dq current control.

**Suggested exercises**

1. Add $R_L = 1$ Ω to the filter inductor (`replace(p, R_L=1.0)`).
   Plot the new plant Bode and the new closed-loop response.
   How does Q change?
2. Implement cross-coupling decoupling: subtract $\omega L_f i_q$
   from the $d$-axis command and add $\omega L_f i_d$ to the $q$-axis
   command. Does it help?
3. Step $V_d^{ref}$ from 187 V to 100 V at $t$ = 50 ms. Plot the
   transient. Does it stay clean?
4. Push the load to imbalance (different $R$ in each phase). The dq
   compensator will struggle because positive- and negative-sequence
   components mix. Production designs add a parallel negative-
   sequence compensator.
